# 🚀 Phase Unwrapping - Swin-UNet Training (Colab)

This notebook facilitates training the **Residue-Aware Swin-UNet** on Google Colab GPUs.

**Prerequisites:**
1.  Upload the `ali_proj` folder to your Google Drive.
2.  Open this notebook in Colab.
3.  Set the runtime to **GPU** (Runtime > Change runtime type > T4/A100).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Navigate to Project Directory (Adjust path as needed)
import os
project_path = '/content/drive/MyDrive/ali_proj'  # <--- CHANGE THIS IF NEEDED
if os.path.isdir(project_path):
    %cd {project_path}
else:
    print(f"Warning: {project_path} not found. Please verify the path.")

In [ ]:
# 3. Install Dependencies
!pip install -e .
!pip install wandb tyro timm  # Ensure extra deps are present
# !pip install h5py  # Usually pre-installed but good to check

In [ ]:
# 4. Login to WandB (Optional but Recommended)
import wandb
wandb.login()

## 🧬 Data Generation

Generate a large dataset directly on Colab (fast I/O due to local generation + HDF5).

In [ ]:
!python -m phase_unwrap.cli generate \
    --out-dir data/colab_train_2k \
    --num-samples 2000 \
    --fmt h5 \
    --shard-size 200

## 🏋️ Training

Start training. Checkpoints will be saved to `runs/colab_run`.
If the runtime disconnects, simply re-run this cell; the script will auto-resume from `runs/colab_run/final.pth` if configured.

In [ ]:
!python -m phase_unwrap.cli train \
    --out-dir runs/colab_run \
    --data-dir data/colab_train_2k \
    --epochs 50 \
    --batch-size 32 \
    --device cuda \
    --override logging.use_wandb=True \
    --override logging.wandb_project=phase_unwrap \
    --override logging.resume_from=runs/colab_run/final.pth

## 🔎 Inference & Visualization

Run inference on the generated test set or a custom folder.

In [ ]:
!python -m phase_unwrap.cli infer \
    --checkpoint runs/colab_run/best.pth \
    --input data/colab_train_2k \
    --output-dir inference_colab \
    --device cuda